# Generate Synthetic Wrought Alloy Database

Uses both **GMM** and **Bayesian GMM (BGMM)** to sample compositions, labels each pool with per-target forward models from config, scores both pools with a balanced metric, and saves the best pool to **synthetic_wrought.csv** for 05_backward_wrought. Run 01 and 02 first.

In [1]:
import os
import warnings

warnings.filterwarnings('ignore')

try:
    from utils import (
        load_hyperparams,
        load_backward_gmm_params,
        load_backward_generator_params,
        load_backward_selection_config,
        get_default_hyperparams,
    )
    from synthetic_generation_core import INPUT_COLS, load_wrought, run_dual_generator_once, env_int
except ImportError:
    import sys
    sys.path.insert(0, os.getcwd())
    from utils import (
        load_hyperparams,
        load_backward_gmm_params,
        load_backward_generator_params,
        load_backward_selection_config,
        get_default_hyperparams,
    )
    from synthetic_generation_core import INPUT_COLS, load_wrought, run_dual_generator_once, env_int

WROUGHT_PATH = 'wrought_alloys_final.csv'
OUTPUT_PATH = os.getenv('SYNTHETIC_OUTPUT_PATH', 'synthetic_wrought.csv')
SCORES_PATH = os.getenv('SYNTHETIC_SCORES_PATH', 'synthetic_wrought_generator_scores.csv')
N_SAMPLES = env_int('GENERATOR_SINGLE_SAMPLES', 50000)
RANDOM_SEED = env_int('GENERATOR_SINGLE_SEED', 42)
print(f'Using RANDOM_SEED={RANDOM_SEED}, N_SAMPLES={N_SAMPLES}')

In [2]:
# Load wrought data (full df with compositions + targets for training models)
df_real, target_list = load_wrought(WROUGHT_PATH) if os.path.exists(WROUGHT_PATH) else (None, [])
if df_real is None:
    raise FileNotFoundError(f'Not found: {WROUGHT_PATH}. Run with wrought data.')
print(f'Loaded wrought data: {df_real.shape}, targets: {len(target_list)}')

Loaded wrought data: (868, 31), targets: 13


In [3]:
# Load generator config for one dual-generator run
wrought_config = load_hyperparams('wrought')
by_target = (wrought_config or {}).get('by_target') or {}
if not by_target:
    raise ValueError('wrought.by_target not in config. Run 01_hyperparameter_tuning_forward.ipynb first.')

gmm_params = load_backward_gmm_params('wrought') or get_default_hyperparams('GMM')
bgmm_params = load_backward_generator_params('wrought', 'BGMM') or get_default_hyperparams('BGMM')
selection_cfg = load_backward_selection_config('wrought') or get_default_hyperparams('BACKWARD_SYNTHETIC_SELECTION')

Sampled 50000 valid compositions.


In [4]:
# Run one seeded dual-generator pass and select best pool
result = run_dual_generator_once(
    df_real=df_real,
    by_target=by_target,
    gmm_params=gmm_params,
    bgmm_params=bgmm_params,
    selection_cfg=selection_cfg,
    get_default_hyperparams=get_default_hyperparams,
    n_samples=N_SAMPLES,
    random_seed=RANDOM_SEED,
)

best_generator = result['best_generator']
gen_df = result['best_df']
scores_df = result['scores']

print('Generator scores:')
print(scores_df)
print(f'Selected generator: {best_generator}')
print(f'Total columns: {list(gen_df.columns)}')

  Predicted UTS (MPa)


  Predicted YS (MPa)


  Predicted Fatigue Strength (MPa)


  Predicted Shear Strength (MPa)


  Predicted Y (GPa)


...

  Predicted TC (W/m-K)


  Predicted TE Coeff


  Predicted Thermal Diffusivity 


  Predicted EC Volume (% IACS)


  Predicted EC Weight (% IACS)
Total columns: ['Al', 'Si', 'Fe', 'Cu', 'Mn', 'Mg', 'Cr', 'Ni', 'Zn', 'Ga', 'V', 'Ti', 'UTS (MPa)', 'YS (MPa)', 'Fatigue Strength (MPa)', 'Shear Strength (MPa)', 'Y (GPa)', 'G (GPa)', 'Density (g/cc)', 'Cp (J/kg-K)', 'TC (W/m-K)', 'TE Coeff', 'Thermal Diffusivity ', 'EC Volume (% IACS)', 'EC Weight (% IACS)']


In [5]:
# Save synthetic pool for backward search + generator score table
gen_df.to_csv(OUTPUT_PATH, index=False)
scores_df.to_csv(SCORES_PATH, index=False)
print(f'Saved {len(gen_df)} rows to {OUTPUT_PATH}')
print(f'Saved generator score table to {SCORES_PATH}')
print(f'Completed with RANDOM_SEED={RANDOM_SEED}, selected={best_generator}')

Saved 50000 rows to synthetic_wrought.csv
